# 💊 Real-World Data Analytics: Patient Adherence & Journeys

This project simulates how to extract patient insights from medical claims and EMR data using Python and SQL.

We'll calculate medication adherence (PDC), model patient journeys, and prepare data for visualization in Tableau or Power BI.

In [1]:
import pandas as pd

# Load data (ensure files are in the same folder as this notebook)
demo = pd.read_csv("demographics.csv")
diag = pd.read_csv("diagnoses.csv")
meds = pd.read_csv("medications.csv")

# Preview datasets
print("Demographics:")
print(demo.head())
print("Diagnoses:")
print(diag.head())
print("Medications:")
print(meds.head())

Demographics:
   patient_id  age  gender   region
0           1   83  Female     West
1           2   73    Male     West
2           3   59    Male  Midwest
3           4   52  Female  Midwest
4           5   65  Female    South
Diagnoses:
   patient_id diagnosis_code diagnosis_date
0           1          E78.5     2022-10-25
1           2            I10     2022-09-20
2           3          E11.9     2022-04-22
3           4          E11.9     2022-03-01
4           5          E11.9     2022-01-02
Medications:
   patient_id      ndc_code     rx_date  days_supply
0           1  0002-8215-01  2022-03-02           60
1           1  0025-1951-60  2022-04-01           90
2           1  0025-1951-60  2022-05-31           30
3           1  0002-8215-01  2022-07-30           30
4           2  0025-1951-60  2022-01-01           30


## 🧭 Step 1: Build Patient Journey Table

In [2]:
# Convert dates
diag["diagnosis_date"] = pd.to_datetime(diag["diagnosis_date"])
meds["rx_date"] = pd.to_datetime(meds["rx_date"])

# Get first Rx date per patient
first_rx = meds.groupby("patient_id")["rx_date"].min().reset_index().rename(columns={"rx_date": "first_rx_date"})

# Merge into journey table
journey = diag.merge(first_rx, on="patient_id").merge(demo, on="patient_id")
journey["days_to_first_rx"] = (journey["first_rx_date"] - journey["diagnosis_date"]).dt.days

journey.head()

,patient_id,diagnosis_code,diagnosis_date,first_rx_date,age,gender,region,days_to_first_rx
0,1,E78.5,2022-10-25,2022-03-02,83,Female,West,-237
1,2,I10,2022-09-20,2022-01-01,73,Male,West,-262
2,3,E11.9,2022-04-22,2022-01-31,59,Male,Midwest,-81
3,4,E11.9,2022-03-01,2022-01-01,52,Female,Midwest,-59
4,5,E11.9,2022-01-02,2022-01-31,65,Female,South,29


## 📊 Step 2: Calculate Medication Adherence (PDC)

In [3]:
# Sum days supplied over 180-day observation period
adherence = meds.groupby("patient_id")["days_supply"].sum().reset_index()
adherence["observation_days"] = 180
adherence["PDC"] = adherence["days_supply"] / adherence["observation_days"]
adherence["adherent"] = adherence["PDC"] >= 0.8

adherence.head()

,patient_id,days_supply,observation_days,PDC,adherent
0,1,210,180,1.166667,True
1,2,210,180,1.166667,True
2,3,150,180,0.833333,True
3,4,390,180,2.166667,True
4,5,90,180,0.500000,False


## 📤 Step 3: Export for Tableau or Power BI

In [4]:
journey.to_csv("journey_export.csv", index=False)
adherence.to_csv("adherence_export.csv", index=False)
print("Export complete. Ready for Tableau!")

Export complete. Ready for Tableau!
